## 1. Import Libraries

We rely on the standard data-science stack plus scikit-learn for the ready-made
implementation and SciPy for the distance computations used in the Elbow Method.

In [ ]:
import pandas as pd                       # data loading and manipulation (DataFrames)
import numpy as np                        # numerical operations and arrays
import matplotlib.pyplot as plt           # plotting and visualisation
from sklearn.cluster import KMeans        # scikit-learn's ready-made K-Means implementation
from sklearn import metrics               # evaluation metrics for clustering
from scipy.spatial.distance import cdist  # pairwise distances (used in the Elbow Method)

## 2. Load and Explore the Dataset

Each row is one mall customer. The key columns we will cluster on are
**Annual Income (k$)** and **Spending Score** (a 1–100 score the mall assigns based on
purchasing behaviour). Below we load the data and take a first look.

In [ ]:
# Read the dataset into a DataFrame and preview the first 5 rows
customer_data = pd.read_csv("../data/Mall_Customers.csv")
customer_data.head()

In [ ]:
# Check for missing values in each column — all zeros means the data is clean
customer_data.isna().sum()

In [ ]:
# Inspect column data types — 'Genre' is categorical (text); the rest are numeric
customer_data.dtypes

In [ ]:
# Scatter plot of the two features we will cluster on
plt.scatter(customer_data['Annual_Income_(k$)'], customer_data['Spending_Score'])
plt.xlabel('Annual_Income_(k$)')
plt.ylabel('Spending_Score')
plt.title('Customers: Annual Income vs. Spending Score')
plt.show()

In [ ]:
# Choose the number of clusters and pick K random data points as the initial centroids
K = 3
centroids = customer_data.sample(n=K)  # randomly sample K rows to act as starting centroids
print(centroids.head())

In [ ]:
# Visualise the initial (randomly chosen) centroids in black against the data points
K = 3
centroids = customer_data.sample(n=K)  # get K random centroids
plt.scatter(customer_data['Annual_Income_(k$)'], customer_data['Spending_Score'])      # data points
plt.scatter(centroids['Annual_Income_(k$)'], centroids['Spending_Score'], c='black')   # centroids
plt.xlabel('Annual_Income_(k$)')
plt.ylabel('Spending_Score')
plt.title('Initial random centroids (black)')
plt.show()

In [ ]:
# The 'Genre' column is categorical text — K-Means needs numeric input, so we inspect it first
customer_data['Genre']

### One-Hot Encoding the categorical column

K-Means works on numeric distances, so the text column `Genre` must be converted to numbers.
**One-hot encoding** turns it into a binary column. We use `drop_first=True` to keep just one
column (`Genre_Male`): `True` for male, `False` for female — which avoids redundant columns.

In [ ]:
# Convert the categorical 'Genre' column into a numeric one-hot encoded column
customer_data = pd.get_dummies(customer_data, columns=['Genre'], drop_first=True)

In [ ]:
# Confirm 'Genre' is now the numeric 'Genre_Male' column
customer_data.head()

In [ ]:
# Build a boolean mask: True where a row's CustomerID is one of the chosen centroids
mask = customer_data['CustomerID'].isin(centroids.CustomerID.tolist())
print(mask.head())
X = customer_data[~mask]  # X holds all points EXCEPT the centroids — these are what we cluster
X.head()

In [ ]:
diff = 1   # tracks how much the centroids move between iterations (loop stops when it hits 0)
j = 0      # iteration counter (used to force the first pass to run)
XD = X
while (diff != 0):
    i = 1
    # STEP 1 — Distance: for each centroid, compute the Euclidean distance to every data point.
    # The list of distances for centroid i is stored as a new column i in X.
    for index1, row_c in centroids.iterrows():
        ED = []
        for index2, row_d in XD.iterrows():
            # Squared difference of each feature between the centroid (row_c) and the point (row_d)
            d1 = (row_c["Annual_Income_(k$)"] - row_d["Annual_Income_(k$)"]) ** 2  # income term
            d2 = (row_c["Spending_Score"] - row_d["Spending_Score"]) ** 2          # spending term
            d = np.sqrt(d1 + d2)   # Euclidean distance = sqrt(sum of squared differences)
            ED.append(d)
        X[i] = ED   # store this centroid's distances as column i
        i = i + 1

    # STEP 2 — Assign clusters: each point joins the cluster of its NEAREST centroid (min distance)
    C = []
    for index, row in X.iterrows():
        min_dist = row[1]   # start by assuming centroid 1 is closest
        pos = 1
        for i in range(K):
            if row[i + 1] < min_dist:   # found a closer centroid
                min_dist = row[i + 1]
                pos = i + 1
        C.append(pos)
    X["Cluster"] = C   # record each point's assigned cluster
    print(X)

    # STEP 3 — Update centroids: the new centroid is the mean position of its cluster's points
    try:
        centroids_new = X.groupby(["Cluster"]).mean()[["Spending_Score", "Annual_Income_(k$)"]]
    except Exception as e:
        print(f'error : {str(e)}')

    # STEP 4 — Check convergence: how far did the centroids move this iteration?
    # On the very first pass we force diff=1 so the loop runs at least twice; after that we
    # measure the total shift in centroid positions. The loop ends once diff == 0 (no movement).
    if j == 0:
        diff = 1
        j = j + 1
    else:
        diff = (centroids_new['Spending_Score'] - centroids['Spending_Score']).sum() + (centroids_new['Annual_Income_(k$)'] - centroids['Annual_Income_(k$)']).sum()
    centroids = X.groupby(["Cluster"]).mean()[["Spending_Score", "Annual_Income_(k$)"]]   # adopt the new centroids

In [ ]:
# Plot each cluster in a different colour, with the final centroids in black
color = ['grey', 'blue', 'orange']
for k in range(K):
    data = X[X["Cluster"] == k + 1]   # points belonging to cluster k+1
    plt.scatter(data["Annual_Income_(k$)"], data["Spending_Score"], c=color[k])
plt.scatter(centroids["Annual_Income_(k$)"], centroids["Spending_Score"], c='black')  # final centroids
plt.xlabel('Annual_Income_(k$)')
plt.ylabel('Spending_Score')
plt.title('Final clusters from the from-scratch K-Means')
plt.show()

In [ ]:
# Fit scikit-learn's KMeans with 3 clusters on the income & spending features
from sklearn.cluster import KMeans
km_sample = KMeans(n_clusters=3)
km_sample.fit(customer_data[["Annual_Income_(k$)", "Spending_Score"]])

In [ ]:
# Visualise the clusters. .labels_ holds the cluster each point was assigned to by KMeans.
import seaborn as sns
labels_sample = km_sample.labels_
customer_data['label'] = labels_sample
sns.scatterplot(x='Annual_Income_(k$)', y='Spending_Score', hue='label', data=customer_data)

In [ ]:
# Cast all column names to strings so scikit-learn accepts them (some are integer column names)
X.columns = X.columns.astype(str)
X.columns

In [ ]:
# Select the numeric features to cluster on for the Elbow and Silhouette analyses
features_for_clustering = ['Age', 'Annual_Income_(k$)', 'Spending_Score']
X_clustering = X[features_for_clustering]
X_clustering.columns = X_clustering.columns.astype(str)
X_clustering.columns

## 5. Choosing the Optimal Number of Clusters

How many clusters should we use? K-Means can't tell us — we have to choose `K`. Two common
techniques help: the **Elbow Method** and the **Silhouette Method**.

### Elbow Method

We fit K-Means for a range of `K` values and measure the **distortion** (the average distance
of points to their assigned centroid). Distortion always falls as `K` grows, but at some point
the improvement levels off — the "elbow" of the curve. That bend suggests a good value of `K`.

In [ ]:
# For each K from 1 to 9, fit KMeans and record the distortion (mean distance to nearest centroid)
distortions = []
K = range(1, 10)
for k in K:
    kmeanModel = KMeans(n_clusters=k)
    kmeanModel.fit(X_clustering)
    # cdist computes the distance from every point to every centroid; we take the nearest one per point
    distortions.append(sum(np.min(cdist(X_clustering, kmeanModel.cluster_centers_, 'euclidean'), axis=1)) / X_clustering.shape[0])

In [ ]:
# Plot distortion vs K — look for the "elbow" where the curve bends and flattens
plt.plot(K, distortions, 'bx-')
plt.xlabel("k")
plt.ylabel("Distortion")
plt.title("The elbow method ")
plt.show()

### Silhouette Method

The **silhouette score** measures how well each point fits its own cluster compared to other
clusters. It ranges from **-1 to 1** — higher is better (points are tight within their cluster
and far from others). We compute the average score for several values of `K`; the `K` with the
highest score is a strong candidate for the best number of clusters.

In [ ]:
# Compute the average silhouette score for K = 2..8 and store each result
from sklearn.metrics import silhouette_score

sil_avg = []
range_n_clusters = [2, 3, 4, 5, 6, 7, 8]

for k in range_n_clusters:
    kmeans = KMeans(n_clusters=k).fit(X_clustering)
    labels = kmeans.labels_   # cluster assignment for each point
    sil_avg.append(silhouette_score(X_clustering, labels, metric='euclidean'))

In [ ]:
# Plot silhouette score vs K — the highest point indicates the best number of clusters
plt.plot(range_n_clusters, sil_avg, 'bx-')
plt.xlabel("Values of K")
plt.ylabel("Silhouette score")
plt.title("Silhouette analysis for optimal k")
plt.show()

In [ ]:
# Silhouette score for K=3 using the Euclidean distance metric (appended to our results list)
kmeans = KMeans(n_clusters=3).fit(X_clustering)
labels = kmeans.labels_
sil_avg.append(silhouette_score(X_clustering, labels, metric='euclidean'))
sil_avg

In [ ]:
# Same K=3, but scored with the Manhattan distance metric to compare against Euclidean
kmeans = KMeans(n_clusters=3).fit(X_clustering)
labels = kmeans.labels_
sil_avg.append(silhouette_score(X_clustering, labels, metric='manhattan'))
sil_avg

In [ ]:
# Assign a new, unseen data point to its nearest cluster using the trained model
new_data_point = np.array([[2, 3, 5]])

new_data_cluster = kmeans.predict(new_data_point)

print("New data point belongs to cluster:", {new_data_cluster[0]})

In [ ]:
# Fit KMeans with 5 clusters on income & spending for the final customer segmentation
from sklearn.cluster import KMeans
km_sample = KMeans(n_clusters=5)
km_sample.fit(customer_data[["Annual_Income_(k$)", "Spending_Score"]])

In [ ]:
# Colour each customer by its assigned cluster to reveal the 5 segments
import seaborn as sns
labels_sample = km_sample.labels_
customer_data['label'] = labels_sample
sns.scatterplot(x='Annual_Income_(k$)', y='Spending_Score',
                hue='label', data=customer_data)

## Summary

- We built **K-Means from scratch** to see how assignment and centroid updates iterate until convergence.
- We reproduced the same idea in **two lines with scikit-learn**.
- The **Elbow** and **Silhouette** methods both help pick a sensible number of clusters.
- The final **5-cluster** segmentation splits mall customers into clear groups by income and spending —
  exactly the kind of insight a business could use for targeted marketing.

**Key takeaway:** K-Means is simple and fast, but you must choose `K` yourself, it's sensitive to the
initial random centroids, and it assumes roughly round, similarly-sized clusters.